# Data Cleaning

Prepare a dataset of completed deliveries with usable event times. Eligibility is decided per order so that every item of a retained order stays in the output.

**Contents**

1. [Project setup and input](#cleaning-setup) — Load the full merge and record its initial dimensions.
2. [Input integrity](#cleaning-integrity) — Check text domains, item keys and repeated attributes.
3. [Cleaning rules](#cleaning-clean) — Normalize city names and select eligible orders.
4. [Timestamp checks](#cleaning-time) — Review missing events, reversals and unusual deadlines.
5. [Whole-order eligibility](#cleaning-eligibility) — Reconcile the exclusions and identify the affected orders.
6. [Retained dataset](#cleaning-retained) — Compare coverage and distributions after filtering.
7. [Save and verify the cleaned dataset](#cleaning-save) — Reload the saved file and verify the original input.

Sections 4–7 use the cleaned table and diagnostics built in section 3. Once these are in memory, each of those sections can be revisited directly.

[User Guide](../docs/Cleaning/User_Guide.md) · [Technical Reference](../docs/Cleaning/Technical_Reference.md) · [Cleaning Report](../reports/Cleaning/Cleaning_Report.md)


<a id="cleaning-setup" name="cleaning-setup"></a>

## 1. Project setup and input

Locate the interim input and processed output, then record a baseline for checking source preservation.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    """Find the repository containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path
    raise FileNotFoundError(
        "Could not locate the project root containing src/ and requirements.txt."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt

from src.cleaning.plots import build_cleaning_visuals
from src.cleaning import (
    CLEANED_DATASET_NAME,
    assert_cleaned_roundtrip,
    clean_order_items,
    format_cleaning_preview,
    inspect_text_fields,
    save_cleaned_data,
)
from src.cleaning.validation import (
    assert_input_unchanged,
    capture_input_baseline,
    validate_cleaning_input,
)
from src.data_loader import load_data
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA
from src.paths import INTERIM_DIR, PROCESSED_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim input: {INTERIM_DIR}")
print(f"Processed output: {PROCESSED_DIR}")


### Load the full merged table


In [ ]:
df = load_data("merged_order_items", stage="interim")
input_baseline = capture_input_baseline(df)
input_summary = pd.DataFrame([
    ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
    ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
], columns=["metric", "reference", "observed"])
input_summary["Matches reference"] = input_summary["reference"].eq(input_summary["observed"])
display(format_cleaning_preview(input_summary))


<a id="cleaning-integrity" name="cleaning-integrity"></a>

## 2. Input integrity

Check identifiers and entity attributes before filtering. A malformed key could group unrelated items or separate items from the same order.


### Text formats and valid codes

Check ID and ZIP formats, state codes and status values before interpreting the records they identify.


In [ ]:
text_inspection = inspect_text_fields(df)
with pd.option_context("display.max_columns", None):
    display(format_cleaning_preview(text_inspection["summary"]))
    if not text_inspection["issues"].empty:
        print("First ten invalid cells; full records: text_inspection['issues']")
        display(format_cleaning_preview(text_inspection["issues"]))


### Item keys and attributes shared across records

Verify item keys and confirm that attributes repeated across items agree within each entity.


In [ ]:
input_validation = validate_cleaning_input(df)
print("Item and entity key profiles")
display(format_cleaning_preview(input_validation["key_profile"]))
print("Entity attribute consistency")
display(format_cleaning_preview(input_validation["entity_consistency"]))


<a id="cleaning-clean" name="cleaning-clean"></a>

## 3. Cleaning rules

EDA found repeated spaces for three sellers and zero weights for four products. Normalize city whitespace and flag nonpositive measurements. Keep existing optional nulls, zero freight and observed price or freight extremes.

Keep delivered orders with four lifecycle timestamps in sequence and an estimated delivery date for comparison. Apply the measurement and shipping-deadline exclusions to the entire order. The following sections inspect the diagnostics returned by this call.


In [ ]:
cleaned_df, cleaning_details = clean_order_items(df)
cleaning_visuals = build_cleaning_visuals(df, cleaned_df, cleaning_details)
# Close pyplot registrations; explicit display below retains the figure objects.
for figure in cleaning_visuals["figures"].values():
    plt.close(figure)


### Whitespace changes in city names before filtering

Compare city strings before and after whitespace normalization so that spelling variants caused by spacing count together.


In [ ]:
text_change_summary = cleaning_details["text_changes"]
display(format_cleaning_preview(text_change_summary))


### Physical measurements before filtering

A physical measurement of zero or less flags an order for exclusion. Existing missing measurements can remain because delivery timing does not require them.


In [ ]:
measurement_summary = cleaning_details["measurements"]
display(format_cleaning_preview(measurement_summary))
print("Preparation and quality-flag validation")
display(format_cleaning_preview(cleaning_details["inspection_checks"]))


<a id="cleaning-time" name="cleaning-time"></a>

## 4. Timestamp checks

Read the timestamp flags on the full input to see which orders would have undefined or negative stage durations and which item deadlines require review.


### Purchase, approval, carrier handoff and delivery

EDA found handoffs recorded before approval. Compare full timestamps so that reversals within the same day are also visible.


In [ ]:
timeline_summary = cleaning_details["timeline"]
core_timestamp_summary = cleaning_details["core_timestamps"]
display(format_cleaning_preview(timeline_summary))
display(format_cleaning_preview(core_timestamp_summary))


### Item shipping deadlines

EDA found four item deadlines in 2020 on purchases from 2017. The review rule flags deadlines before purchase or beyond the last observed purchase year.


In [ ]:
deadline_summary = cleaning_details["deadlines"]
deadline_review_records = cleaning_details["deadline_records"]
display(format_cleaning_preview(deadline_summary))
if not deadline_review_records.empty:
    print("First ten review items; full records: deadline_review_records")
    with pd.option_context("display.max_columns", None):
        display(format_cleaning_preview(deadline_review_records))


<a id="cleaning-eligibility" name="cleaning-eligibility"></a>

## 5. Whole-order eligibility

Compare independent reason counts with sequential removals. A single item can disqualify an order, and one order can meet several exclusion rules.


### Orders affected by each rule

Read each count against the full input. The same order may appear under several reasons.


In [ ]:
filter_reasons = cleaning_details["filter_reasons"]
display(format_cleaning_preview(filter_reasons))


### Sequential exclusions

Follow the remaining order and item counts after each rule to obtain the total removed.


In [ ]:
filter_flow = cleaning_details["filter_flow"]
with pd.option_context("display.max_columns", None):
    display(format_cleaning_preview(filter_flow))


### Order retention by filtering step

The lower panel shows newly excluded orders at each step.


In [ ]:
display(cleaning_visuals["figures"]["filtering"])


### Excluded order records

Use the reason flags to trace an individual exclusion back to the rule or rules it met.


In [ ]:
excluded_orders = cleaning_details["excluded_orders"]
print(f"Excluded orders: {len(excluded_orders):,}; first ten shown")
with pd.option_context("display.max_columns", None):
    display(format_cleaning_preview(excluded_orders))


<a id="cleaning-retained" name="cleaning-retained"></a>

## 6. Retained dataset

Compare the retained cohort with the full input, then inspect its measurements, calendar coverage and geographic composition.


### Before-and-after dimensions

Compare counts at each unit of observation, from individual items to distinct sellers.


In [ ]:
display(format_cleaning_preview(cleaning_details["dataset_summary"]))


### Missing values before and after cleaning

Each rate uses the item count at that stage. For grouped physical fields, the count applies to each field individually.


In [ ]:
missingness_review = cleaning_visuals["tables"]["missingness_display"]
display(format_cleaning_preview(
    missingness_review[["attribute", "missing_before", "missing_after", "missing_pct_after"]],
    rows=max(1, len(missingness_review)),
))
display(cleaning_visuals["figures"]["missingness"])


### Prices, freight and product measurements

Price and freight use item rows. Weight and dimensions use distinct products. The lower panels show the 1st to 99th percentiles.


In [ ]:
numeric_review = cleaning_visuals["tables"]["numeric"]
display(format_cleaning_preview(
    numeric_review[["column", "grain", "unit", "rows", "missing", "min", "median", "max"]],
))
display(cleaning_visuals["figures"]["numeric"])


### Orders by purchase month and approval hour

Compare monthly purchase volume with the distribution of recorded approval hours at order level.


In [ ]:
display(cleaning_visuals["figures"]["calendar"])


### Product categories by number of distinct products

Count each product once. The chart groups categories outside the ten most common under Other and shows missing categories separately.


In [ ]:
display(cleaning_visuals["figures"]["categories"])


### Customer and seller states

Each customer record and each seller counts once in its respective panel.


In [ ]:
display(cleaning_visuals["figures"]["geography"])


### Validate retained orders and item values

Check that retained orders kept all their items and that changes to retained values follow the cleaning rules.


In [ ]:
display(format_cleaning_preview(cleaning_details["output_checks"]))
assert_input_unchanged(df, input_baseline)


### Cleaned order-item preview

Inspect a sample of retained items with the display labels and formats used throughout this notebook.


In [ ]:
cleaning_preview = format_cleaning_preview(cleaned_df, rows=10)
display(cleaning_preview)


<a id="cleaning-save" name="cleaning-save"></a>

## 7. Save and verify the cleaned dataset

Write `cleaned_order_items` to `data/processed` and read back the format actually saved. Compare values, dtypes and row order before reusing the dataset.


In [ ]:
saved_path = save_cleaned_data(cleaned_df)
reloaded_df = load_data(
    CLEANED_DATASET_NAME,
    stage="processed",
    file_format=saved_path.suffix.lstrip("."),
)
assert_cleaned_roundtrip(cleaned_df, reloaded_df)
print(f"Verified processed output: {saved_path}")


### Verify the input table is unchanged

Compare the original merged table with the baseline taken before cleaning.


In [ ]:
assert_input_unchanged(df, input_baseline)
print("Source preservation and processed roundtrip checks passed.")
